# Opening range: continuation, rejection, retests, and repeat trades

Exploratory baseline. Run top to bottom in Colab. Uses one-minute stock bars for honest event ordering; resamples completed 2- and 5-minute candles. No QQQ, sector, gap, or weekday filter. All times Eastern. This notebook does not contain backtest results.

Definitions: 15-minute OR is 09:30–09:45; 30-minute OR is 09:30–10:00. Continuation: first close beyond OR on a 2- or 5-minute candle, or first 1-minute close beyond OR. Rejection: 2-minute candle wicks beyond an OR boundary and closes back inside. Retest: after a close beyond the OR, a later 2-minute candle touches the broken boundary and closes back beyond it. One active position per ticker and configuration; later entries require a fresh setup after exit. All signals enter at next available 1-minute open, never at the signal candle close. Baseline 1% stop, 2% target; VWAP rejection also reports VWAP as an alternate target. Intrabar double touches count stop first. Close remaining positions at 15:55.

Retest diagnostics distinguish all breakout signals, winning breakouts, and breakouts that retested before their original trade hit target/stop. Each configuration is independent; overlapping configurations are comparison alternatives, not a combined tradable portfolio. Candles with missing minutes are excluded from signals, and each day must have a complete opening range. Quotes, spread and borrow availability are absent from OHLCV; slippage is a configurable estimate.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, re
from IPython.display import display
drive.mount('/content/drive')
ROOT = next((p for p in [Path('/content/drive/MyDrive/backtesting'),Path('/content/drive/MyDrive/Backtesting')] if p.exists()),None)
if ROOT is None: raise FileNotFoundError('Locate the Backtesting folder and set ROOT to it')
OUT = Path('/content/drive/MyDrive/Opening_Range_Phase1_Results')
OUT.mkdir(parents=True,exist_ok=True)
OR_MINUTES=(15,30)
STOP_PCT=.01
TARGET_PCT=.02
SLIPPAGE_BPS=5  # per side; change to reflect your actual fills
LAST_ENTRY='15:00'
EXIT_TIME='15:55'
EXCLUDE={'QQQ','SPY','IWM','^VIX','SOXX','XLK','XLV','XLI','XLY','XLF','XLC','XLE','XLB','XLP','XLRE','XLU'}
print('Data:',ROOT,'Output:',OUT)

In [ ]:
def read_stock_days(root):
    files=list(root.rglob('*_1m_*.csv'))
    if not files: raise FileNotFoundError('No *_1m_*.csv under '+str(root))
    groups={}
    for f in files:
        m=re.match(r'(.+)_1m_\d{4}-\d{2}-\d{2}',f.stem)
        if not m: continue
        ticker=m.group(1).upper()
        if ticker in EXCLUDE: continue
        try:
            d=pd.read_csv(f)
            datecol=next((c for c in d.columns if c.lower() in ('datetime','date','timestamp')),d.columns[0])
            t=pd.to_datetime(d[datecol],errors='coerce',utc=True)
            if t.notna().sum()==0: continue
            d['time']=t.dt.tz_convert('America/New_York')
            for c in ['Open','High','Low','Close','Volume']: d[c]=pd.to_numeric(d[c],errors='coerce')
            d=d.dropna(subset=['time','Open','High','Low','Close'])
            d=d.set_index('time')[['Open','High','Low','Close','Volume']]
            d=d.between_time('09:30','15:59')
            for day,g in d.groupby(d.index.date):
                groups.setdefault((ticker,str(day)),[]).append(g)
        except Exception as e: print('SKIP',f.name,str(e)[:120])
    for (ticker,day),frames in sorted(groups.items()):
        d=pd.concat(frames).sort_index()
        d=d[~d.index.duplicated(keep='last')]
        yield ticker,day,d
def make_bars(d,n):
    # anchor to 09:30; only complete, consecutive n-minute candles
    origin=d.index[0].normalize()+pd.Timedelta(hours=9,minutes=30)
    bars=d.resample(f'{n}min',origin=origin,label='left',closed='left').agg(
        {'Open':'first','High':'max','Low':'min','Close':'last','Volume':'sum'})
    counts=d['Close'].resample(f'{n}min',origin=origin,label='left',closed='left').count()
    return bars[counts.eq(n)].dropna(subset=['Open','Close'])
def vwap_series(d):
    typical=(d.High+d.Low+d.Close)/3
    cumvol=d.Volume.fillna(0).cumsum()
    return (typical*d.Volume.fillna(0)).cumsum()/cumvol.replace(0,np.nan)
def valid_day(d):
    expected=pd.date_range(d.index[0].normalize()+pd.Timedelta(hours=9,minutes=30),periods=30,freq='min',tz='America/New_York')
    return set(expected).issubset(set(d.index)) and d.index.is_unique

In [ ]:
def signals_for_day(d,or_minutes):
    open_time=d.index[0].normalize()+pd.Timedelta(hours=9,minutes=30)
    end=open_time+pd.Timedelta(minutes=or_minutes)
    opening=d.loc[(d.index>=open_time)&(d.index<end)]
    if len(opening)!=or_minutes: return [],None
    hi=float(opening.High.max()); lo=float(opening.Low.min())
    bars2=make_bars(d,2); bars5=make_bars(d,5)
    vwap=vwap_series(d)
    # close timestamp is start + candle length; open of next minute follows signal
    out=[]
    for n,bars in [(1,d),(2,bars2),(5,bars5)]:
        prev_state={'long':False,'short':False}
        for start,r in bars.iterrows():
            close_time=start+pd.Timedelta(minutes=n)
            if start<end or close_time>pd.Timestamp(str(start.date())+' '+LAST_ENTRY,tz='America/New_York'): continue
            # no signal on incomplete/missing data; n=1 always a full minute
            if n==1 and start not in d.index: continue
            if r.Close>hi and not prev_state['long']:
                out.append(dict(time=close_time,kind=f'break_{n}m',side='long',boundary=hi,level=hi))
            if r.Close<lo and not prev_state['short']:
                out.append(dict(time=close_time,kind=f'break_{n}m',side='short',boundary=lo,level=lo))
            prev_state={'long':r.Close>hi,'short':r.Close<lo}
            if n==2:
                if r.High>hi and lo<=r.Close<=hi:
                    out.append(dict(time=close_time,kind='reject_2m',side='short',boundary=hi,level=float(vwap.asof(start))))
                if r.Low<lo and lo<=r.Close<=hi:
                    out.append(dict(time=close_time,kind='reject_2m',side='long',boundary=lo,level=float(vwap.asof(start))))
    # retest must follow a prior 2m breakout; require close beyond boundary
    armed={'long':False,'short':False}
    for start,r in bars2.iterrows():
        t=start+pd.Timedelta(minutes=2)
        if start<end: continue
        if armed['long'] and r.Low<=hi and r.Close>hi:
            out.append(dict(time=t,kind='retest_2m',side='long',boundary=hi,level=hi))
            armed['long']=False
        if armed['short'] and r.High>=lo and r.Close<lo:
            out.append(dict(time=t,kind='retest_2m',side='short',boundary=lo,level=lo))
            armed['short']=False
        if r.Close>hi: armed['long']=True
        if r.Close<lo: armed['short']=True
        if r.Close<=hi: armed['long']=False
        if r.Close>=lo: armed['short']=False
    return sorted(out,key=lambda x:x['time']),(hi,lo)
def next_open(d,signal_time):
    # a bar timestamped exactly at close begins after the signal is known
    loc=d.index.searchsorted(signal_time)
    if loc>=len(d): return None
    t=d.index[loc]
    if t>signal_time+pd.Timedelta(minutes=1): return None
    return t

In [ ]:
def simulate(d,signals,bounds,or_minutes,ticker,day):
    rows=[]
    for kind in ['break_1m','break_2m','break_5m','retest_2m','reject_2m']:
      for side in ['long','short']:
       pool=[s for s in signals if s['kind']==kind and s['side']==side]
       busy_until=d.index[0]
       used=set()
       number=0
       for s in pool:
        if s['time']<=busy_until: continue
        entry_t=next_open(d,s['time'])
        if entry_t is None or entry_t.strftime('%H:%M')>LAST_ENTRY: continue
        if entry_t in used: continue
        # Reject only toward VWAP, provided target lies on profitable side and within OR
        if kind=='reject_2m':
            if not np.isfinite(s['level']): continue
            if side=='short' and not (s['level']<d.loc[entry_t,'Open'] and s['level']>=bounds[1]): continue
            if side=='long' and not (s['level']>d.loc[entry_t,'Open'] and s['level']<=bounds[0]): continue
        entry=float(d.loc[entry_t,'Open'])*(1+(SLIPPAGE_BPS/10000)*(1 if side=='long' else -1))
        stop=entry*(1-STOP_PCT if side=='long' else 1+STOP_PCT)
        target=entry*(1+TARGET_PCT if side=='long' else 1-TARGET_PCT)
        end_t=pd.Timestamp(day+' '+EXIT_TIME,tz='America/New_York')
        path=d.loc[(d.index>=entry_t)&(d.index<=end_t)]
        if path.empty: continue
        exit_t=path.index[-1]; exit_raw=float(path.iloc[-1].Close); outcome='time'
        vwap_hit=False; vwap_time=None
        for t,r in path.iterrows():
            if kind=='reject_2m' and not vwap_hit and ((side=='long' and r.High>=s['level']) or (side=='short' and r.Low<=s['level'])):
                vwap_hit=True;vwap_time=t
            loss=(r.Low<=stop if side=='long' else r.High>=stop)
            win=(r.High>=target if side=='long' else r.Low<=target)
            if loss or win:
                exit_t=t;exit_raw=stop if loss else target
                outcome='stop' if loss else 'target'
                break
        exit_price=exit_raw*(1-(SLIPPAGE_BPS/10000)*(1 if side=='long' else -1))
        pnl=(exit_price-entry)/entry*(1 if side=='long' else -1)
        number+=1;used.add(entry_t)
        rows.append(dict(ticker=ticker,date=day,weekday=entry_t.day_name(),or_minutes=or_minutes,
            or_high=bounds[0],
            or_low=bounds[1],
            kind=kind,side=side,trade_number=number,signal_time=s['time'],entry_time=entry_t,
            exit_time=exit_t,entry=entry,exit=exit_price,outcome=outcome,
            pnl_pct=100*pnl,R=pnl/STOP_PCT,vwap_target=s['level'] if kind=='reject_2m' else np.nan,
            vwap_hit=vwap_hit,vwap_hit_time=vwap_time))
        busy_until=exit_t+pd.Timedelta(minutes=1)
    return rows

In [ ]:
trades=[]; coverage=[]
for i,(ticker,day,d) in enumerate(read_stock_days(ROOT),1):
    if not valid_day(d):
        coverage.append(dict(ticker=ticker,date=day,status='incomplete opening 30m'));continue
    coverage.append(dict(ticker=ticker,date=day,status='ok',bars=len(d)))
    for n in OR_MINUTES:
        signals,bounds=signals_for_day(d,n)
        if bounds:
            trades.extend(simulate(d,signals,bounds,n,ticker,day))
    if i%100==0: print('Days processed:',i,'Trades:',len(trades))
ledger=pd.DataFrame(trades)
coverage=pd.DataFrame(coverage)
if ledger.empty: raise RuntimeError('No trades found; inspect coverage and ticker data paths')
ledger.to_csv(OUT/'trade_ledger.csv',index=False)
coverage.to_csv(OUT/'coverage.csv',index=False)
display(coverage.status.value_counts())
display(ledger.head())

In [ ]:
def summarize(df,by):
    return df.groupby(by,dropna=False).agg(trades=('R','size'),wins=('outcome',lambda x:(x=='target').sum()),
       stops=('outcome',lambda x:(x=='stop').sum()),mean_R=('R','mean'),
       net_R=('R','sum'),median_R=('R','median'),vwap_hit=('vwap_hit','sum')).reset_index()
summary=summarize(ledger,['or_minutes','kind','side'])
repeat=summarize(ledger,['or_minutes','kind','side','trade_number'])
weekday=summarize(ledger,['or_minutes','kind','weekday'])
ticker=summarize(ledger,['or_minutes','kind','ticker'])
for name,table in [('summary',summary),('repeat_trades',repeat),('weekday',weekday),('ticker',ticker)]:
    table.to_csv(OUT/(name+'.csv'),index=False)
    print('\n'+name);display(table.head(40))

## Retest timing diagnostic

The next cell checks a separate question: for each filled 2-minute breakout, does price return to the broken OR boundary **after entry but before that trade exits**? It reports this rate for all breakouts and target winners. It also compares trades actually entered on the retest in the summary above. A bar that touches both the boundary and stop can be ambiguous; this diagnostic labels the touch, while the trade engine treats the stop first.

In [ ]:
def retest_diagnostic(root,breaks):
    lookup={(t,day):d for t,day,d in read_stock_days(root)}
    result=[]
    for row in breaks.itertuples():
        d=lookup.get((row.ticker,row.date))
        if d is None: continue
        boundary=row.or_high if row.side=='long' else row.or_low
        span=d.loc[(d.index>row.entry_time)&(d.index<=row.exit_time)]
        touched=span.Low<=boundary if row.side=='long' else span.High>=boundary
        times=span.index[touched]
        first=times[0] if len(times) else pd.NaT
        result.append(dict(ticker=row.ticker,date=row.date,or_minutes=row.or_minutes,
            side=row.side,outcome=row.outcome,entry_time=row.entry_time,exit_time=row.exit_time,
            retested=bool(len(times)),first_retest=first,
            minutes_to_retest=(first-row.entry_time).total_seconds()/60 if len(times) else np.nan))
    return pd.DataFrame(result)
breaks=ledger[ledger.kind=='break_2m']
retests=retest_diagnostic(ROOT,breaks)
retests.to_csv(OUT/'breakout_retest_diagnostic.csv',index=False)
rates=retests.groupby(['or_minutes','side','outcome']).agg(
    breakout_trades=('retested','size'),retested=('retested','sum'),
    retest_rate=('retested','mean'),median_minutes=('minutes_to_retest','median')).reset_index()
rates.to_csv(OUT/'retest_rates.csv',index=False)
display(rates)
print('Results saved in',OUT)
print('Review the target-winner retest rate alongside all-breakout retest rate and retest-entry fills.')